# Module 02 — Vector Search

**What we build in this notebook:**
- Embed text using an ONNX model (no PyTorch)
- Compute cosine similarity between a query and a document
- Chunk documents and search them by hand with dot products
- Use minsearch's `VectorSearch` as a proper search library
- Compare text search vs vector search on a real vocabulary mismatch
- Combine both with Reciprocal Rank Fusion (hybrid search)

**Why ONNX instead of sentence-transformers?**  
`sentence-transformers` pulls in PyTorch (~4.8 GB).  
ONNX Runtime is ~147 MB. Same vectors. No GPU needed. Runs anywhere.

---
Run `python download.py` once before this notebook to fetch the model.

## Q1 — Embedding a Query

**The goal:** turn a text string into a vector of numbers.

The shape `(384,)` confirms the right model loaded.  
The first value is a deterministic fingerprint — same input → same output every time.

**Expected answer: `-0.02`**

In [ ]:
from embedder import Embedder

# Load the ONNX model (from ./models/Xenova/all-MiniLM-L6-v2)
embed = Embedder()

# Quick sanity check — embed a test string, check the shape
v = embed.encode("Hello world")
print("Sanity check shape:", v.shape)   # should be (384,)

In [ ]:
# The actual Q1 query
query = "How does approximate nearest neighbor search work?"

query_vector = embed.encode(query)

print(f"Embedding shape : {query_vector.shape}")   # (384,)
print(f"First value     : {query_vector[0]}")       # -0.02058203437252893

# ANSWER: rounded to 2 decimal places → -0.02

### What just happened?

The `Embedder.encode()` pipeline:
1. **Tokenize** — split the sentence into subword tokens, convert to integer IDs
2. **Inference** — feed the IDs through the ONNX model, get a hidden state per token
3. **Pool** — take the mean of all token hidden states (ignoring padding tokens)
4. **Normalize** — scale the vector to length 1.0 (L2 norm)

Step 4 is what makes `dot product == cosine similarity` — see Q2.

---
## Q2 — Cosine Similarity

**The goal:** embed one document, compute how similar it is to the query.

We use the SQLite vector search lesson as the target document.  
The query is about ANN search; the document covers vector search broadly + SQLite details.  
A moderate similarity (~0.36) is expected because the doc's embedding is diluted by off-topic content.

**Expected answer: `0.37`**

In [ ]:
from gitsource import GithubRepositoryDataReader

# Read all lesson .md files from a specific commit of the course repo
# commit_id pins the data so results are always reproducible
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(f"Total documents: {len(documents)}")   # 72

In [ ]:
# Each document is a dict with 'filename' and 'content' keys
print(documents[0].keys())
print(documents[0]['filename'])

In [ ]:
# Find the specific document we want to compare against
sqlite_doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

print("Found:", sqlite_doc["filename"])
print(f"Content length: {len(sqlite_doc['content'])} characters")

In [ ]:
# Embed the whole document as one vector
doc_vector = embed.encode(sqlite_doc["content"])

# Cosine similarity = dot product (because both vectors are L2-normalized)
# query_vector · doc_vector  →  cos(angle between them)
similarity = query_vector.dot(doc_vector)

print(f"Cosine similarity: {similarity}")
# ANSWER: ~0.361 → rounded to 0.37

### Why ~0.36 and not higher?

The full lesson covers: SQLite, SQL tables, persistence, implementation details, AND vector search.  
Embedding the whole document averages across all those topics.  
The resulting vector is pulled away from the specific ANN concept we asked about.

**This is the motivation for chunking** — see Q3.

---
## Q3 — Chunking and Search by Hand

**The goal:** split documents into overlapping windows, embed each chunk independently,
then find the best chunk using a manual dot product.

**Expected answer:** best chunk filename = `07-sqlitesearch-vector.md`  
Max score ≈ `0.65` (vs 0.36 from the whole document — nearly double!)

In [ ]:
from gitsource import chunk_documents

# size=2000: each chunk is max 2000 characters
# step=1000: the window slides 1000 chars each time
#            → consecutive chunks OVERLAP by 1000 chars
#            → no passage is silently split at a chunk boundary
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(f"Total chunks: {len(chunks)}")   # 295

In [ ]:
# Each chunk has: filename, start (character offset), content
print(chunks[0].keys())
print("Filename:", chunks[0]['filename'])
print("Start   :", chunks[0]['start'])
print("Content preview:", chunks[0]['content'][:200])

In [ ]:
# Embed ALL chunks in one batch call
# encode_batch is efficient — one ONNX inference call for the whole batch
texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(texts)

print(f"Matrix shape: {X.shape}")   # (295, 384)

In [ ]:
# BRUTE-FORCE VECTOR SEARCH — this one line IS the entire search operation
#
# X          shape: (295, 384)  — one row per chunk
# query_vector shape: (384,)    — the query
#
# X.dot(query_vector) computes ONE dot product per row simultaneously
# Result: (295,) — one similarity score per chunk

scores = X.dot(query_vector)

print(f"Scores shape : {scores.shape}")   # (295,)
print(f"Best score   : {scores.max()}")   # ~0.649  ← vs 0.36 for the whole doc!

In [ ]:
# argmax() returns the INDEX of the highest score
best_idx = scores.argmax()

print(f"Best chunk index    : {best_idx}")                        # 94
print(f"Best chunk filename : {chunks[best_idx]['filename']}")   # 07-sqlitesearch-vector.md
print(f"Best chunk start    : {chunks[best_idx]['start']}")      # 1000 (character offset)

# The SAME document won — but the focused chunk scored 0.65 vs 0.36 for the full doc
# Chunking improved retrieval quality without changing the model at all

### The key insight

The same document won in Q2 and Q3. The lesson was always the right answer.  
The problem in Q2 was that the whole-doc embedding was too diluted to score well.  
Chunking gave the relevant passage its own focused vector — and it scored nearly double.

**Chunking often matters more than model choice.**

---
## Q4 — Vector Search with minsearch

**The goal:** use `VectorSearch` from minsearch instead of manual dot products.  
The library handles indexing, ranking, and returning payloads.  
Notice: `.search()` takes a **vector**, not raw text — that's intentional (see note below).

**Expected answer:** `04-evaluation/lessons/05-search-metrics.md`

In [ ]:
from minsearch import VectorSearch

# Create the index
vector_index = VectorSearch()

# fit() takes:
#   X      — the (295, 384) matrix of chunk embeddings
#   chunks — the list of dicts (the 'payload' attached to each vector)
vector_index.fit(X, chunks)

In [ ]:
# New query — about evaluation metrics
query = "What metric do we use to evaluate a search engine?"

# YOU must embed the query — VectorSearch doesn't know about text
query_vector = embed.encode(query)

results = vector_index.search(query_vector)

# The top result
print("Top result filename:", results[0]['filename'])
# 04-evaluation/lessons/05-search-metrics.md  ← the lesson literally about search metrics!

In [ ]:
# Look at the content of the top result
print(results[0]['content'][:500])

### Why does `.search()` take a vector, not text?

`VectorSearch` is responsible for **indexing and ranking**.  
The embedding model is responsible for **text → vector conversion**.  
Keeping them separate means you can swap either one without touching the other.  

Want to switch from `all-MiniLM-L6-v2` to OpenAI embeddings?  
Change the `embed` object. The `vector_index` code stays identical.

---
## Q5 — Text Search vs Vector Search

**The goal:** run the same query through keyword search AND vector search.  
Observe what vector search finds that text search misses (vocabulary mismatch).

**Expected answer:** vector search finds `08-pgvector.md` as top result; text search misses it entirely.

In [ ]:
from minsearch import Index

# Build a keyword (TF-IDF) index over the same chunks
text_index = Index(
    text_fields=["content"],      # fields to search in
    keyword_fields=["filename"]   # fields for exact filtering
)
text_index.fit(chunks)

In [ ]:
# The query uses words that DON'T appear in the pgvector lesson
query = "How do I store vectors in PostgreSQL?"

# --- TEXT SEARCH ---
# Matches documents containing the query's exact words
text_results = text_index.search(query=query, num_results=5)

print("TEXT SEARCH results:")
for r in text_results:
    print(" ", r['filename'])

In [ ]:
# --- VECTOR SEARCH ---
# Matches documents with similar MEANING regardless of exact words
query_vector = embed.encode(query)
vector_results = vector_index.search(query_vector, num_results=5)

print("VECTOR SEARCH results:")
for r in vector_results:
    print(" ", r['filename'])

### What happened?

**Text search** found lessons containing the words "vectors" and "PostgreSQL" — but missed `08-pgvector.md`  
because that lesson uses different vocabulary: "pgvector", "pg_vector extension", "storing embeddings".

**Vector search** found `08-pgvector.md` at the top — because both the query and the lesson  
express the **same concept**, so their vectors point in similar directions.

This is the **vocabulary mismatch problem**, and it's the primary reason vector search exists.

Neither method is wrong — they have different strengths:

| | Text Search | Vector Search |
|---|---|---|
| **Good at** | Exact terms, error codes, IDs | Paraphrasing, synonyms, meaning |
| **Bad at** | Vocabulary mismatch | Rare domain-specific exact terms |

---
## Q6 — Hybrid Search with Reciprocal Rank Fusion (RRF)

**The goal:** combine text + vector results so that documents strong in BOTH methods win.

**Expected answer:** `01-agentic-rag/lessons/13-function-calling.md` ranks first.  
It was 2nd in text search and 5th in vector search — never first in either individually.

### How RRF works

For each document in each result list:
```
score += 1 / (k + rank)   where rank starts at 0
```
- `k=60` is a smoothing constant from the original 2009 paper
- A document appearing in TWO lists accumulates contributions from both
- Raw scores are discarded — only position (rank) matters
- This solves the scale mismatch between TF-IDF weights and cosine similarities

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    """
    Reciprocal Rank Fusion.

    Args:
        result_lists: list of result lists, each from a different retrieval method
        k: smoothing constant (default 60, from the original RRF paper)
        num_results: how many results to return

    Returns:
        list of docs, ranked by combined RRF score
    """
    scores = {}   # key -> accumulated RRF score
    docs   = {}   # key -> original doc dict

    for results in result_lists:
        for rank, doc in enumerate(results):
            # Use (filename, start) as a unique key for each chunk
            key = (doc["filename"], doc["start"])

            # Add this rank's contribution
            # rank=0 → 1/(60+0)=0.0167, rank=4 → 1/(60+4)=0.0156
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    # Sort by accumulated score, highest first
    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query = "How do I give the model access to tools?"

# Get results from both methods
text_results   = text_index.search(query, num_results=5)
query_vector   = embed.encode(query)
vector_results = vector_index.search(query_vector, num_results=5)

# Print each method's ranking to see the disagreement
print("TEXT SEARCH top 5:")
for i, doc in enumerate(text_results):
    print(f"  {i}  {doc['filename']}  start={doc['start']}")

print()

print("VECTOR SEARCH top 5:")
for i, doc in enumerate(vector_results):
    print(f"  {i}  {doc['filename']}  start={doc['start']}")

In [ ]:
# Combine with RRF — pass both result lists
hybrid_results = rrf([vector_results, text_results])

print("HYBRID (RRF) top 5:")
for i, doc in enumerate(hybrid_results):
    print(f"  {i+1}  {doc['filename']}  start={doc['start']}")

# 13-function-calling.md rises to #1 despite never being first in either individual method
# It was the only chunk that ranked HIGH IN BOTH lists

### Why RRF works

`13-function-calling.md` ranked **2nd in text** and **5th in vector**.  
No other chunk appeared high in both lists.  
RRF accumulated contributions from both — total score ≈ `1/(60+1) + 1/(60+4)` — more than any single-list leader.

The consensus signal beat the individual top picks.

**One important rule:** don't tune `k=60` without a proper evaluation set (Module 4).  
It's the default from the original paper and works well across most retrieval tasks.

---

## Summary

| Q | What you learned |
|---|---|
| 1 | Embedding converts text to a fixed-length vector; `(384,)` = correct model |
| 2 | Cosine similarity = dot product (for normalized vectors); ~0.36 = related but diluted |
| 3 | Chunking alone doubled the best score (0.36 → 0.65) — same model, better doc prep |
| 4 | Libraries wrap the dot product; interface separation makes systems swappable |
| 5 | Text search matches words; vector search matches meaning; vocabulary mismatch is real |
| 6 | RRF combines ranked lists (not raw scores); consensus beats individual best picks |

**The big takeaway:**  
Choosing between text/vector/hybrid search is an empirical question.  
Module 4 gives you the tools to measure which actually works for your data.